In [ ]:
import torch
import os
import subprocess

repo_root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"]).decode().strip()
os.chdir(repo_root)

print("cwd =", os.getcwd())

from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from models.VAE.default import UNetVAE
from train.VAE.train_VAE_unet_subsampled_dataset import load_sigma_dataset


def show_slices_plotly(
    target: torch.Tensor,
    recon: torch.Tensor,
    xq: np.ndarray,
    yq: np.ndarray,
    zq: np.ndarray,
    sigma_min: float,
    max_points: int,
    generated: torch.Tensor,
) -> None:
    target = target.detach().cpu().squeeze(0).squeeze(0).numpy()
    recon = recon.detach().cpu().squeeze(0).squeeze(0).numpy()
    generated_np = generated.detach().cpu().squeeze(0).squeeze(0).numpy()

    zz, yy, xx = np.meshgrid(zq, yq, xq, indexing="ij")
    xx = xx.reshape(-1)
    yy = yy.reshape(-1)
    zz = zz.reshape(-1)

    flat_target = target.reshape(-1)
    sigma_cutoff = max(sigma_min, float(np.quantile(flat_target, 0.05)))
    shared_mask = flat_target > sigma_cutoff
    if not np.any(shared_mask):
        shared_mask = flat_target > sigma_min

    def sample_points(
        vol: np.ndarray,
        mask: np.ndarray | None = None,
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        sigma = vol.reshape(-1)
        mask = (mask if mask is not None else sigma > sigma_min)
        x = xx[mask]
        y = yy[mask]
        z = zz[mask]
        sigma = sigma[mask]
        if sigma.size == 0:
            return x, y, z, sigma
        if sigma.size > max_points:
            idx = np.linspace(0, sigma.size - 1, max_points).astype(int)
            x = x[idx]
            y = y[idx]
            z = z[idx]
            sigma = sigma[idx]
        return x, y, z, sigma

    x_t, y_t, z_t, s_t = sample_points(target, shared_mask)
    x_r, y_r, z_r, s_r = sample_points(recon, shared_mask)
    x_g, y_g, z_g, s_g = sample_points(generated_np, shared_mask)

    color_min = float(s_t.min())
    color_max = float(s_t.max())

    fig = make_subplots(
        rows=1,
        cols=3,
        specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
        subplot_titles=("target", "recon", "prior sample"),
    )

    fig.add_trace(
        go.Scatter3d(
            x=x_t,
            y=y_t,
            z=z_t,
            mode="markers",
            marker=dict(
                size=2,
                color=s_t,
                coloraxis="coloraxis",
                opacity=0.7,
            ),
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter3d(
            x=x_r,
            y=y_r,
            z=z_r,
            mode="markers",
            marker=dict(size=2, color=s_r, coloraxis="coloraxis", opacity=0.7),
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Scatter3d(
            x=x_g,
            y=y_g,
            z=z_g,
            mode="markers",
            marker=dict(size=2, color=s_g, coloraxis="coloraxis", opacity=0.7),
        ),
        row=1,
        col=3,
    )

    fig.update_layout(
        height=600,
        width=1400,
        coloraxis=dict(
            cmin=color_min,
            cmax=color_max,
            colorscale="Plasma",
            colorbar=dict(title="log(sigma)"),
        ),
    )
    fig.show()


ckpt_path = Path(
    "saved_runs/vae_sigma_unet_beta1e-02_kl600/checkpoints/best_val_loss.pt"
)
device = "cuda:1"

model = UNetVAE(base_channels=8, latent_channels=8).to(device)

ckpt = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

sigma, xq, yq, zq = load_sigma_dataset(Path("data/3dert_test2.npy"), log_clamp_min=1e-12)
sigma = sigma.to(device)

sample = sigma[:1]
with torch.no_grad():
    recon, mu, logvar = model(sample)

    recon_mu = model.decode(mu)

    prior_z = torch.randn_like(mu)
    prior_sample = model.decode(prior_z)

show_slices_plotly(
    sample,
    recon,
    xq,
    yq,
    zq,
    sigma_min=float(torch.log(torch.tensor(1e-12))),
    max_points=8000,
    generated=prior_sample,
)


cwd = /home/johnma/ert3d_test2
